In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pzest.config import load_config
from pzest.dataset.splits import load_splits
from pzest.dataset.dataset import get_dataloader
from pzest.pipelines.train_pipeline import main as run_training
from pzest.pipelines.evaluate_pipeline import main as run_evaluate
from pzest.evaluation.calibrate import find_temperature, apply_temperature

## Train and Evaluate DeepZ

Trains DeepZ on the preprocessed GalaxiesML-Spectra dataset, evaluates on the held-out test set, and visualises training history and results.
Prerequisites: `02_preprocessing.ipynb` should be run first.

In [ ]:
config = load_config("../config/default.yaml")
config.paths.figures.mkdir(parents=True, exist_ok=True)

Train

In [ ]:
history = run_training()

Training history plots

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

ax1.plot(history["epoch"], history["train_loss"], label="Train")
ax1.plot(history["epoch"], history["val_loss"],   label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("CRPS Loss")
ax1.set_title("Training and Validation Loss")
ax1.legend()

ax2.plot(history["epoch"], history["val_sigma_nmad"])
ax2.set_xlabel("Epoch")
ax2.set_ylabel("σ_NMAD")
ax2.set_title("Validation σ_NMAD")

ax3.plot(history["epoch"], history["lr"])
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Learning Rate")
ax3.set_title("Learning Rate Schedule")
ax3.set_yscale("log")

plt.tight_layout()
plt.savefig(config.paths.figures / "training_history.png", dpi=150)
plt.show()

Evaluate

In [ ]:
results, model, device = run_evaluate()

Save Evaluation Results as JSON

In [ ]:
results_to_save = {
    "model":          "modelX",
    "best_epoch":     results["best_epoch"],
    "val_sigma_nmad": results["val_snmad"],
    "sigma_nmad":     float(results["sigma_nmad"]),
    "bias":           float(results["bias"]),
    "outlier_rate":   float(results["outlier_rate"]),
    "crps":           float(results["crps"]),
    "notes":          "...",
}

results_path = config.paths.checkpoints_dir.parent / "modelX_results.json"
with open(results_path, "w") as f:
    json.dump(results_to_save, f, indent=2)
print(f"Results saved to {results_path}")

Calibrate temperature on validation set

In [ ]:
# Calibrate temperature on validation set
_, val_indices, _ = load_splits(config)

bin_edges = np.linspace(
    config.model.redshift_min,
    config.model.redshift_max,
    config.model.num_bins + 1,
)

In [ ]:
val_loader_cal = get_dataloader(
    hdf5_path=config.paths.processed_hdf5_file,
    indices=val_indices,
    bin_edges=bin_edges,
    config=config,
    shuffle=False,
)

T = find_temperature(model, val_loader_cal, device)
print(f"Optimal temperature: {T:.4f}")
apply_temperature(model, T)

# Re-evaluate with calibrated model
results_cal, _, _ = run_evaluate()

Scatter plot

In [ ]:
z_true = results["z_true"]
z_pred = results["point_estimates"]

fig, ax = plt.subplots(figsize=(6, 6))
ax.hexbin(z_true, z_pred, gridsize=100, cmap="viridis", mincnt=1)
ax.plot(
    [config.model.redshift_min, config.model.redshift_max],
    [config.model.redshift_min, config.model.redshift_max],
    "r--", lw=1, label="1:1"
)
ax.set_xlabel("Spectroscopic redshift")
ax.set_ylabel("Photometric redshift")
ax.set_title(
    f"DeepZ  |  "
    f"σ_NMAD={results['sigma_nmad']:.4f}  |  "
    f"bias={results['bias']:.4f}  |  "
    f"outliers={results['outlier_rate']:.4f}"
)
ax.legend()
plt.tight_layout()
plt.savefig(config.paths.figures / "scatter_zpred_ztrue.png", dpi=150)
plt.show()

PIT Histograms

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(results["pit"], bins=50, density=True, edgecolor="none")
ax1.axhline(1.0, color="red", linestyle="dashed", label="Uniform (ideal)")
ax1.set_xlabel("PIT value")
ax1.set_ylabel("Density")
ax1.set_title(f"Uncalibrated  |  CRPS={results['crps']:.4f}")
ax1.legend()

ax2.hist(results_cal["pit"], bins=50, density=True, edgecolor="none")
ax2.axhline(1.0, color="red", linestyle="dashed", label="Uniform (ideal)")
ax2.set_xlabel("PIT value")
ax2.set_ylabel("Density")
ax2.set_title(f"Calibrated (T={T:.4f})  |  CRPS={results_cal['crps']:.4f})")
ax2.legend()

plt.tight_layout()
plt.savefig(config.paths.figures / "pit_calibration_comparison.png", dpi=150)
plt.show()

PIT by redshift range

In [ ]:
z_true_test = results["z_true"]
pit_values = results["pit"]

low_z_mask = z_true_test < 0.5
mid_z_mask = (z_true_test >= 0.5) & (z_true_test < 1.5)
high_z_mask = z_true_test >=1.5

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

masks = [low_z_mask, mid_z_mask, high_z_mask]
titles = ["z < 0.5", "0.5 ≤ z < 1.5", "z ≥ 1.5"]

for ax, mask, title in zip(axes, masks, titles):
    ax.hist(pit_values[mask], bins=50, density=True, edgecolor="none")
    ax.axhline(1.0, color="r", linestyle="--", label="Uniform (ideal)")
    ax.set_xlabel("PIT value")
    ax.set_ylabel("Density")
    ax.set_title(f"{title} (n={mask.sum():,})")
    ax.legend()

plt.suptitle("PIT histogram by redshift range", y=1.02)
plt.tight_layout()
plt.savefig(config.paths.figures / "pit_by_redshift.png", dpi=150, bbox_inches="tight")
plt.show()

Mean PDF across all test galaxies

In [ ]:
pdfs = results["pdfs"]
bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Mean PDF across all galaxies
mean_pdf = pdfs.mean(axis=0)

# Mean PDF by redshift range
mean_pdf_low = pdfs[low_z_mask].mean(axis=0)
mean_pdf_mid = pdfs[mid_z_mask].mean(axis=0)
mean_pdf_high = pdfs[high_z_mask].mean(axis=0)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(bin_centres, mean_pdf)
axes[0].set_xlabel("Redshift")
axes[0].set_ylabel("Mean P(z)")
axes[0].set_title("All galaxies")

for ax, mean, title in zip(
        axes[1:],
        [mean_pdf_low, mean_pdf_mid, mean_pdf_high],
        ["z < 0.5", "0.5 ≤ z < 1.5", "z ≥ 1.5"]
):
    ax.plot(bin_centres, mean)
    ax.set_xlabel("Redshift")
    ax.set_ylabel("Mean P(z)")
    ax.set_title(title)

plt.suptitle("Mean predicted PDF by redshift range", y=1.02)
plt.tight_layout()
plt.savefig(config.paths.figures / "mean_pdf_by_redshift.png", dpi=150, bbox_inches="tight")
plt.show()